In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score, mean_absolute_error, confusion_matrix

In [ ]:
#Previsão de tempo estimado de espera
df = pd.read_csv('/content/bd_tmp_espera2.csv')
df.head(5)

In [ ]:
# EXIBINDO INFORMAÇÕES
print("="*60)
print("DATASET CRIADO COM SUCESSO!")
print("="*60)
print(f"\nTotal de registros: {len(df)}")
print(f"\nColunas: {list(df.columns)}")
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nPrimeiras 10 linhas:")
print(df.head(10))
print("\nEstatísticas do tempo de espera:")
print(df['tempo_de_espera_min'].describe())

In [12]:
x = df.drop('tempo_de_espera_min', axis=1)
y = df['tempo_de_espera_min'].values

In [13]:

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), ['horario', 'dia_semana', 'tipo_consulta', 'horario_pico']),
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), ['pacientes_na_fila_antes', 'pacientes_agendados_no_mesmo_horario',
             'tempo_medio_consulta_tipo_min', 'medico_habito_atraso'])
    ],
    remainder='drop'
)

In [20]:
X_transform = preprocessor.fit_transform(x)

In [21]:
X_Train, X_Test, y_Train, y_Test = train_test_split(X_transform, y, test_size=0.3, random_state=42)

In [22]:
paramgrid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [1.0, 'sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [ ]:
rfr = RandomForestRegressor()
random_search = RandomizedSearchCV(
    estimator=rfr,
    param_distributions=paramgrid,
    n_iter=10,  # Number of parameter settings that are sampled
    cv=5,  # Number of cross-validation folds
    verbose=2,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1  # Use all available processors
)
# Fit the RandomizedSearchCV instance
random_search.fit(X_Train, y_Train)

# Retrieve the best parameters found by RandomizedSearchCV
best_params = random_search.best_params_
best_params

In [ ]:
#Melhorar as features
random_forest_regressor = RandomForestRegressor(**best_params)
random_forest_regressor.fit(X_Train, y_Train)
test_predic = random_forest_regressor.predict(X_Test)
mse = mean_squared_error(y_Test, test_predic)
r2 = r2_score(y_Test, test_predic)
mae = mean_absolute_error(y_Test, test_predic)
print(mse)
print(r2)
print(mae)

In [ ]:
#plot de grafico com matplotlib
z = np.polyfit(y_Test, test_predic, 1)
p = np.poly1d(z)
plt.figure(figsize=(10, 6))
plt.scatter(y_Test, test_predic, alpha=0.6)
plt.plot(y_Test, p(y_Test), color='red', linestyle='--')
plt.xlabel('Valores Reais')
plt.ylabel('Valores Previstos')

In [25]:
df_to_predict = pd.read_csv('/content/bd_tmp_espera_to_predict.csv')

In [ ]:
df_to_predict_transform = preprocessor.transform(df_to_predict)
predict = random_forest_regressor.predict(df_to_predict_transform)
df_to_predict['tempo_de_espera_min'] = predict
df_to_predict.head(5)

In [ ]:
df_to_predict.to_csv('bd_tmp_espera_predict.csv', index=False)

In [ ]:
#Algoritmo de classificação
df_classifier = pd.read_csv('/content/data_to_train_classifier.csv')
df_classifier.head(5)

In [ ]:
df_classifier['histórico_de_atrasos'].value_counts()

In [ ]:
# EXIBINDO INFORMAÇÕES
print("="*60)
print("DATASET CRIADO COM SUCESSO!")
print("="*60)
print(f"\nTotal de registros: {len(df_classifier)}")
print(f"\nColunas: {list(df_classifier.columns)}")
print("\nTipos de valores das colunas")
print(df_classifier.dtypes)
print("\nPrimeiras 10 linhas:")
print(df_classifier.head(10))
print("\nEstatísticas do tempo de espera:")
print(df_classifier['histórico_de_atrasos'].describe())

In [52]:
preprocessor_classifier = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), ['consulta', 'tipo_de_transporte', 'dia_semana', 'horario_da_consulta', 'horario_pico', 'PcD']
),
        ('num', Pipeline(steps=[
            ('scaler', StandardScaler())
        ]), ['idade', 'distacia_local_de_partida_e_consultorio_km'])
    ],
    remainder='drop'
)

In [53]:
x = df_classifier.drop('histórico_de_atrasos', axis=1)
y = df_classifier['histórico_de_atrasos'].values

label_encoder = LabelEncoder()
X_transformed = preprocessor_classifier.fit_transform(x)
Y_transformed = label_encoder.fit_transform(y)

In [54]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_transformed, Y_transformed)

In [65]:
#X_train, X_test, y_train, y_test  = train_test_split(X_transformed, Y_transformed, test_size=0.3, random_state=42)
X_train, X_test, y_train, y_test  = train_test_split(X_balanced, y_balanced, test_size=0.3, random_state=42)

In [56]:
rfc = RandomForestClassifier()

In [57]:
parameters = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30],
    'min_samples_leaf': [1, 2, 4, 5],
    'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'max_features': [1.0, 'sqrt', 'log2', None],
    'max_samples': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'bootstrap': [True, False]
}

In [ ]:
random_searchCV = RandomizedSearchCV(
    estimator=rfc,
    param_distributions=parameters,
    n_iter=10,
    cv=5,
    verbose=2,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1
)
random_searchCV.fit(X_train, y_train)
best_params_to_rfc = random_searchCV.best_params_
best_params_to_rfc

In [74]:
Random_Forest_Classifier = RandomForestClassifier(**best_params_to_rfc)
Random_Forest_Classifier.fit(X_train, y_train)
y_pred = Random_Forest_Classifier.predict(X_test)
conf_matrix = confusion_matrix(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average='weighted')
accuracy = accuracy_score(y_test, y_pred)
print(conf_matrix)
print(f1Score)
print(accuracy)

[[40 19 13]
 [17 35 18]
 [21 19 31]]
0.4965421531766973
0.49765258215962443


In [ ]:

#Previsão de dados com algoritmo de classificação
df_classifier_predict  = pd.read_csv('/content/data_to_classifier_classify.csv')
df_classifier_predict

In [ ]:
classify_predict = preprocessor_classifier.transform(df_classifier_predict)
predict_class = Random_Forest_Classifier.predict(classify_predict)
predictions_string = label_encoder.inverse_transform(predict_class)
df_classifier_predict['histórico_de_atrasos'] = predictions_string
df_classifier_predict.head(50)

In [ ]:
df_classifier_predict['histórico_de_atrasos'].value_counts()
